# Tutorial 8: QSAR Modeling with scikit-learn

Build a miniature quantitative structure–activity relationship (QSAR) model using RDKit fingerprints and scikit-learn.


## Objectives

- Assemble a labelled dataset of example molecules.
- Convert the structures into Morgan fingerprints for machine-learning friendly descriptors.
- Train and evaluate a logistic regression classifier with standard metrics.


In [ ]:
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split


## Build a Toy Dataset

The dataset below marks simple aromatics and heteroatom-containing molecules as active (1) or inactive (0).


In [ ]:
dataset = pd.DataFrame({
    'smiles': ['CCO', 'CCN', 'CCCl', 'c1ccccc1', 'CC(=O)O', 'CCS', 'c1ccncc1', 'CCOCC', 'c1ccc(Cl)cc1', 'CC(=O)Nc1ccccc1'],
    'active': [0, 0, 1, 1, 0, 1, 1, 0, 1, 1]
})
dataset


## Generate Morgan Fingerprints

Morgan fingerprints are circular fingerprints that summarise the neighbourhood around each atom.


In [ ]:
radius = 2
n_bits = 1024
fps = []
for smiles in dataset['smiles']:
    mol = Chem.MolFromSmiles(smiles)
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    arr = np.zeros((1,), dtype=int)
    DataStructs.ConvertToNumpyArray(fp, arr)
    fps.append(arr)
X = np.array(fps)
y = dataset['active'].values
X.shape


## Train/Test Split and Model Training

We keep 30% of the data for validation and fit a logistic regression classifier on the remaining molecules.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
model = LogisticRegression(max_iter=2000)
model.fit(X_train, y_train)
y_prob = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)
{
    'roc_auc': roc_auc_score(y_test, y_prob),
    'accuracy': accuracy_score(y_test, y_pred)
}


## Inspect the Confusion Matrix

A quick look at the confusion matrix reveals how the classifier performs on the hold-out set.


In [ ]:
cm = confusion_matrix(y_test, y_pred)
pd.DataFrame(cm, index=['actual_inactive', 'actual_active'], columns=['pred_inactive', 'pred_active'])


## Highlight Influential Bits

The absolute coefficient magnitudes correspond to fingerprint bits that the model finds important.


In [ ]:
coeffs = pd.Series(model.coef_[0])
coeffs.abs().sort_values(ascending=False).head(10)
